# LC meshes: figures & exploration

Thin wrapper over the `lc_mesh` library. It reproduces the paper figures **from the
meshes this capsule generates** (the reproducible core + nine percentile meshes), and
lets you explore the mesh-generation parameters. There is no reference to any
previously distributed mesh.

Run `code/reproduce_meshes.py` (or `code/run`) first to write the meshes to `results/`;
the figure cells below load them. All real logic lives in `code/lc_mesh/`.

In [1]:
import sys, pathlib
import numpy as np
import trimesh
sys.path.insert(0, str(pathlib.Path.cwd()))          # so `import lc_mesh` works from code/
sys.path.insert(0, str(pathlib.Path.cwd() / 'code'))  # or from the repo root
import lc_mesh
from lc_mesh import figures

# Resolve inputs/outputs: CodeOcean mounts if present, else local paths.
CAP = pathlib.Path('/root/capsule/data')
if CAP.exists():
    RAW = CAP / 'LC_H2B_trailmap_probabilities_and_point_calls' / 'segmentation_and_quantification'
    TRAILMAP = CAP / 'LC_H2B_trailmap_probabilities_and_point_calls'
    SMARTSPIM = CAP / 'SmartSPIM_807324_2025-08-25_11-34-40_stitched_2025-10-23_17-35-23'
    CCF = CAP / 'ccf_meshes'
    MESH_DIR = pathlib.Path('/root/capsule/results')   # where reproduce_meshes.py wrote the meshes
    RESULTS = pathlib.Path('/root/capsule/results')
else:
    REPO = pathlib.Path.cwd()
    RAW = None                                          # set to your local raw point-call dir
    TRAILMAP = SMARTSPIM = CCF = None
    MESH_DIR = REPO / 'results' / 'reproduced'
    RESULTS = REPO / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

# Optional CCF brain outline for the 3D figures (rendered only if the mesh is found).
brain_mesh = None
if CCF and CCF.exists():
    objs = list(CCF.rglob('997.obj')) or list(CCF.rglob('*.obj'))
    if objs:
        brain_mesh = trimesh.load(objs[0])
print('brain outline:', brain_mesh is not None)

OSError: libgomp.so.1: cannot open shared object file: No such file or directory

## 1. Build LC points from the raw point calls

In [ ]:
df = lc_mesh.build_lc_points(str(RAW))
print(len(df), 'LC points;  columns:', list(df.columns))

## 2. Load the generated meshes
Loads the core + percentile meshes written by `reproduce_meshes.py`. If they are not
present, regenerates them from `df` (slower).

In [ ]:
def _load_or_make(thresh=None):
    name = 'new_core_mesh' if thresh is None else f'percentile_{thresh}'
    p = MESH_DIR / f'{name}.obj'
    if p.exists():
        return trimesh.load(p)
    maker = lc_mesh.make_core_mesh if thresh is None else (lambda d: lc_mesh.make_percentile_mesh(d, thresh))
    return maker(df)[0]

core_mesh = _load_or_make()
percentile_meshes = {t: _load_or_make(t) for t in lc_mesh.config.PERCENTILE_THRESHOLDS}
print('core:', len(core_mesh.vertices), 'verts;  percentiles:', sorted(percentile_meshes))

## 3. Point-in-mesh membership
Several figures color/aggregate points by which mesh contains them. This adds an
`in_*` column per mesh.

**Slow (minutes):** containment tests ~35k points against all ten meshes.

In [ ]:
meshes = {'new_core_mesh': core_mesh, **percentile_meshes}
df = lc_mesh.count_points_in_meshes(df, meshes, verbose=True)
print('added columns:', [c for c in df.columns if c.startswith('in_')])

## 4. Figures (from the generated meshes)

### 4a. kNN minimum-projection heatmaps

In [ ]:
figures.plot_knn_min_projection(df, save_path=str(RESULTS / 'kNN_min_projection_heatmaps.png'))

### 4b. 3D mesh renderings

In [ ]:
figures.plot_percentile_and_core(percentile_meshes, core_mesh, brain_mesh,
                                 save_path=str(RESULTS / 'percentile_90_plus_core.html')).show()
figures.plot_all_percentile_meshes(percentile_meshes, brain_mesh,
                                   save_path=str(RESULTS / 'all_percentiles.html')).show()
figures.plot_points_core_membership(df, percentile_meshes, core_mesh, brain_mesh,
                                    save_path=str(RESULTS / 'points_core_membership.html')).show()

### 4c. Per-sample point counts by hemisphere

In [ ]:
figures.plot_percentile_counts_by_hemisphere(df, save_path=str(RESULTS / 'counts_by_hemisphere.png'))

### 4d. Coronal slices with per-sample PC2 histograms

In [ ]:
_ = figures.plot_coronal_slices_with_pc2(df, core_mesh, save_dir=str(RESULTS / 'slice_svgs'))
print('saved coronal slice SVGs')

### 4e. Raw-image max-projection overlay
Needs the SmartSPIM_807324 zarr volume + raw points (CodeOcean only).

In [ ]:
if SMARTSPIM and SMARTSPIM.exists():
    import zarr
    sample = 'SmartSPIM_807324_2025-08-25_11-34-40_stitched_2025-10-23_17-35-23'
    pts = np.load(TRAILMAP / sample / 'points_raw_807324.npy')
    zarr_vol = zarr.open(str(CAP / sample / 'image_tile_fusing/OMEZarr/Ex_488_Em_525.zarr'), mode='r')['1']
    fig, _ = figures.plot_max_proj_with_points(zarr_vol, pts, ap_range=(3100, 3200))
    fig.show()
else:
    print('SmartSPIM volume not mounted; skipping raw-image overlay.')

## 5. Explore mesh-generation parameters

### 5a. kNN-percentile threshold explorer
The core mesh uses the 67th percentile as its shell cutoff; drag to see the effect.

In [ ]:
import ipywidgets as widgets
widgets.interact(
    lambda threshold: figures.plot_threshold_scatter(df, threshold),
    threshold=widgets.FloatSlider(value=67, min=0, max=100, step=1, continuous_update=False),
)

### 5b. Regenerate a mesh with tweaked parameters
Every parameter is in `lc_mesh.config`. Example: rebuild the 90th-percentile mesh and
view it. Pass `repair_overrides={...}` to `make_percentile_mesh` to try different repair
settings without editing `config.py`.

In [ ]:
mesh90, _ = lc_mesh.make_percentile_mesh(df, 90, verbose=True)
figures.plot_mesh_3d(mesh90, title='percentile 90 (regenerated)').show()